# Visualising the `whest` Datasets

The **ARC White-Box Estimation Challenge 2026** asks a sharp question: given only
the weights of a deep ReLU MLP, what is its mean activation under a standard
normal input? Writing one network's forward map as a product of nonlinear layers,

$$z_0 = x, \qquad z_\ell = \mathrm{relu}(z_{\ell-1} W_\ell), \qquad \ell = 1 \dots L,$$

the target is the deterministic functional

$$F(W) \;=\; \mathbb{E}_{x \sim N(0, I_d)}\big[\, z_L \,\big] \;\in\; \mathbb{R}^{d}.$$

The input distribution is **integrated out**, so $F$ is a function of $W$ alone — a
map $\mathbb{R}^{L \times d \times d} \to \mathbb{R}^{d}$. Nothing is random once
the weights are fixed.

Viewed as a dynamical system, each network is a *product of random matrices acting
on a distribution*, and $F(W)$ is a functional of its terminal distribution. That
framing explains everything this notebook plots: the heavy tail of $F$, the
coordinates that die as depth grows, and why a cheap estimator's error is
structured rather than diffuse.

**This notebook loads everything with `local=True`** — the whest datasets are not
in the published remote snapshot, so they must be read from `data/processed/`.

## Contents

1. [What is on disk](#1)
2. [One network is a sequence of matrices](#2)
3. [The forward convention is `z @ W`, not `z @ W.T`](#3)
4. [The heavy tail of $F$](#4)
5. [The ReLU cone collapses with depth](#5)
6. [Width scaling at fixed depth](#6)
7. [The cheap baseline and its residual](#7)
8. [What a learned model has to beat](#8)

In [2]:
import json
import math

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
from plotly.subplots import make_subplots

from generatedata.load_data import data_names, dataset_info, load_data_as_sequence

# Ground truth is a ~1e-8 quantity here, and TF32 truncates matmul inputs to 10
# mantissa bits (a ~1e-3 relative error per layer), so it must stay off.
torch.backends.cuda.matmul.allow_tf32 = False

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

### Plot styling

One validated palette for the whole notebook, so colour always means the same
thing. Colour has three different *jobs* here and each gets its own treatment:

| job | encoding |
| --- | --- |
| identity (which series) | categorical hues, assigned in fixed order |
| magnitude (how much) | one hue, light → dark |
| polarity (which sign) | two hues with a **neutral grey** midpoint at zero |

The categorical hues below are colourblind-safe as an ordered set; the diverging
ramp is used only where zero is meaningful (signed weights, signed residuals).

In [3]:
# Categorical slots, assigned in fixed order and never cycled.
C_BLUE, C_ORANGE, C_AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK_SOFT, GRID = "#0b0b0b", "#52514e", "#e6e5e1"

# Magnitude: a single hue, light -> dark.
SEQ_BLUE = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
# Polarity: two hues, neutral grey at zero.  Used with zmid=0.
DIV_BLUE_RED = [
    [0.0, "#184f95"], [0.25, "#6da7ec"], [0.5, "#f0efec"],
    [0.75, "#f08a89"], [1.0, "#b02b2b"],
]

LAYOUT = dict(
    template="plotly_white",
    font=dict(family="system-ui, -apple-system, Segoe UI, sans-serif",
              size=13, color=INK),
    paper_bgcolor="#fcfcfb",
    plot_bgcolor="#fcfcfb",
    margin=dict(l=70, r=30, t=60, b=60),
    xaxis=dict(gridcolor=GRID, zerolinecolor=GRID, linecolor=GRID, ticks="outside",
               tickcolor=GRID, title_font_color=INK_SOFT, tickfont_color=INK_SOFT),
    yaxis=dict(gridcolor=GRID, zerolinecolor=GRID, linecolor=GRID, ticks="outside",
               tickcolor=GRID, title_font_color=INK_SOFT, tickfont_color=INK_SOFT),
    legend=dict(bgcolor="rgba(0,0,0,0)", borderwidth=0, font_color=INK_SOFT),
    hoverlabel=dict(font_size=12),
)


def style(fig, title=None, height=420, **kwargs):
    """Apply the shared layout, letting a caller override any part of it.

    Nested dicts are merged rather than replaced, so passing e.g.
    ``legend=dict(orientation="h")`` keeps the shared legend colours.
    """
    layout = {**LAYOUT}
    for key, value in kwargs.items():
        if isinstance(value, dict) and isinstance(layout.get(key), dict):
            layout[key] = {**layout[key], **value}
        else:
            layout[key] = value
    fig.update_layout(**layout, height=height)
    if title:
        fig.update_layout(title=dict(text=title, font=dict(size=15, color=INK), x=0.01))
    return fig

<a id="1"></a>
## 1. What is on disk

Only `whest_w8_d8` is generated by default. The rest of the ladder is opt-in
(`scripts/generatedata_local.py --whest` or `--whest-xl`), so this notebook adapts
to whatever is present rather than assuming a particular set.

Each dataset records what its labels can support, which is the part worth reading
before trusting any number computed from it:

- **`label_mc_se2`** — the variance of the labels' own Monte-Carlo noise. Nothing
  smaller than this is measurable from the dataset.
- **`ut_final_layer_mse`** — the raw MSE of the cheap `UT_fixed` baseline that
  ships with every dataset. This is the number a learned model has to beat.
- **headroom** = the ratio of those two. It says how much room there is between
  "what we can resolve" and "what we are trying to improve on".

In [4]:
names = sorted(n for n in data_names(local=True) if n.startswith("whest"))
if not names:
    raise RuntimeError(
        "No whest datasets found. Generate at least the core one with:\n"
        "    uv run python scripts/generatedata_local.py"
    )

info = {n: dataset_info(n, local=True) for n in names}
inventory = pd.DataFrame([
    {
        "dataset": n,
        "width": i["width"],
        "depth": i["depth"],
        "networks": i["num_points"],
        "labels": ("official 1e9" if i["source"].startswith("official")
                   else f"ours 2^{round(math.log2(i['mc_samples']))}"),
        "label_mc_se2": i["label_mc_se2"],
        "ut_mse": i["ut_final_layer_mse"],
        "headroom": i["ut_final_layer_mse"] / i["label_mc_se2"],
        "dead_coords": i["dead_coordinate_fraction"],
        "dead_nets": i["dead_network_fraction"],
    }
    for n, i in info.items()
]).sort_values(["depth", "width"]).reset_index(drop=True)

print(f"{len(names)} whest dataset(s) available locally\n")
inventory.style.format({
    "label_mc_se2": "{:.2e}", "ut_mse": "{:.3e}", "headroom": "{:,.0f}x",
    "dead_coords": "{:.3f}", "dead_nets": "{:.2f}", "networks": "{:,d}",
})

8 whest dataset(s) available locally



,dataset,width,depth,networks,labels,label_mc_se2,ut_mse,headroom,dead_coords,dead_nets
0,whest_w8_d8,8,8,"10,000",ours 2^18,1.50e-06,2.167e-03,"1,440x",0.199,0.00
1,whest_w16_d8,16,8,"10,000",ours 2^24,1.89e-08,1.387e-03,"73,436x",0.065,0.00
2,whest_w256_d8,256,8,45,official 1e9,1.74e-10,1.310e-04,"751,034x",0.002,0.00
3,whest_w16_d32,16,32,"2,700",ours 2^24,8.65e-09,5.614e-04,"64,907x",0.384,0.00
4,whest_w32_d32,32,32,701,ours 2^24,1.14e-08,5.135e-04,"45,131x",0.302,0.00
5,whest_w64_d32,64,32,178,ours 2^24,6.60e-09,2.919e-04,"44,232x",0.255,0.00
6,whest_w128_d32,128,32,44,ours 2^24,4.04e-09,9.954e-05,"24,628x",0.221,0.00
7,whest_w256_d32,256,32,11,official 1e9,3.66e-11,2.837e-05,"775,091x",0.177,0.00


Two dataset choices for the rest of the notebook: the **smallest** one for anything
that costs real compute (re-running the forward map), and the **deepest, widest**
one for structural plots, which only read labels and are cheap at any size.

In [5]:
def cost(n):
    """Bytes of weights per dataset -- the right proxy for 'expensive to compute on'."""
    i = info[n]
    return i["depth"] * i["width"] ** 2 * i["num_points"]


SMALL = min(names, key=cost)
BIG = max(names, key=lambda n: (info[n]["depth"], info[n]["width"]))
print(f"cheap demos : {SMALL:22s} width {info[SMALL]['width']:3d}  depth {info[SMALL]['depth']:2d}  "
      f"{info[SMALL]['num_points']:,} networks")
print(f"structure   : {BIG:22s} width {info[BIG]['width']:3d}  depth {info[BIG]['depth']:2d}  "
      f"{info[BIG]['num_points']:,} networks")

cheap demos : whest_w8_d8            width   8  depth  8  10,000 networks
structure   : whest_w256_d32         width 256  depth 32  11 networks


<a id="2"></a>
## 2. One network is a sequence of matrices

`load_data_as_sequence` returns the weights as `(networks, depth, width²)` — one
$d \times d$ matrix per timestep, in layer order. That is the honest shape for this
problem: the forward pass is an **ordered** product of operators, so the layer axis
is a sequence axis, not an exchangeable feature axis.

The arrays are **memory-mapped**, so this costs no RAM until you index into them —
which matters, since the largest dataset is 471 MB of weights.

In [6]:
W_seq, F = load_data_as_sequence(BIG, local=True)
width, depth = info[BIG]["width"], info[BIG]["depth"]
W = W_seq.reshape(-1, depth, width, width)     # a view; still on disk

print(f"{BIG}")
print(f"  W_seq {W_seq.shape}  {W_seq.dtype}  ({type(W_seq).__name__} — not yet read)")
print(f"  as matrices {W.shape}   labels F {F.shape}")
print(f"  He init: std {np.asarray(W[0]).std():.6f}  vs  sqrt(2/width) = {math.sqrt(2 / width):.6f}")

whest_w256_d32
  W_seq (11, 32, 65536)  float32  (memmap — not yet read)
  as matrices (11, 32, 256, 256)   labels F (11, 256)
  He init: std 0.088278  vs  sqrt(2/width) = 0.088388


Weights are **signed**, so they call for a diverging ramp with a neutral midpoint
at zero — blue for negative, red for positive, grey for nothing. A sequential ramp
here would imply that zero is an extreme rather than the centre.

Every layer looks statistically identical — iid $N(0, 2/d)$, no biases. All the
structure in $F$ comes from the *product*, not from any individual matrix.

In [7]:
show_layers = [0, depth // 2, depth - 1]
fig = make_subplots(rows=1, cols=len(show_layers), horizontal_spacing=0.06,
                    subplot_titles=[f"layer {l + 1}" for l in show_layers])
lim = float(np.abs(np.asarray(W[0, show_layers])).max())
for k, layer in enumerate(show_layers):
    fig.add_trace(
        go.Heatmap(z=np.asarray(W[0, layer]), colorscale=DIV_BLUE_RED, zmid=0,
                   zmin=-lim, zmax=lim, showscale=(k == len(show_layers) - 1),
                   colorbar=dict(title="w", thickness=12, len=0.9, outlinewidth=0),
                   hovertemplate="row %{y}, col %{x}<br>w = %{z:.4f}<extra></extra>"),
        row=1, col=k + 1,
    )
fig.update_yaxes(autorange="reversed", scaleanchor=None)
style(fig, f"Weight matrices of one network — {BIG}, network 0", height=340)
fig.update_annotations(font=dict(size=13, color=INK_SOFT))
fig.show()

<a id="3"></a>
## 3. The forward convention is `z @ W`, not `z @ W.T`

This is the single most expensive detail to get wrong, so it is worth *checking*
rather than trusting. Re-run the forward map by Monte Carlo on the smallest
dataset and compare against the stored labels.

The transposed convention is plotted alongside. It is not a subtle difference — it
is wrong by an $O(1)$ factor, which is exactly why a plot makes it obvious.

In [8]:
Ws_seq, Fs = load_data_as_sequence(SMALL, local=True)
w_s, d_s = info[SMALL]["width"], info[SMALL]["depth"]
# np.array() copies out of the read-only memmap, which torch requires.
Ws = np.array(Ws_seq[:24], dtype=np.float32).reshape(-1, d_s, w_s, w_s)
Wt = torch.from_numpy(Ws).double()

M = 200_000
gen = torch.Generator().manual_seed(0)
x = torch.randn(M, w_s, generator=gen, dtype=torch.float64)


def forward_mean(weights, transpose=False):
    """E_x[relu-chain(x)] by Monte Carlo, with the per-coordinate variance."""
    z = x
    for layer in range(weights.shape[0]):
        A = weights[layer].T if transpose else weights[layer]
        z = torch.relu(z @ A)
    return z.mean(dim=0).numpy(), z.var(dim=0).numpy()


mc, var = (np.stack(a) for a in zip(*(forward_mean(Wt[i]) for i in range(len(Wt)))))
mc_T = np.stack([forward_mean(Wt[i], transpose=True)[0] for i in range(len(Wt))])
stored = np.asarray(Fs[:len(Wt)], dtype=np.float64)

# Both sides are Monte-Carlo estimates, so judge the difference on the scale of its
# own standard error.  Use the *per-coordinate* activation variance for both terms:
# ours over M draws, and the stored label's over its own mc_samples.  F is heavy
# tailed, so a large coordinate is far noisier than a typical one -- pooling the
# variance (as the dataset's single `label_mc_se2` figure does) would overstate the
# error bar on typical coordinates and understate it on the big ones.
se = np.sqrt(var / M + var / info[SMALL]["mc_samples"])

# A dead coordinate has zero variance, so it has no error bar: both sides must be
# *exactly* zero, and that is a check in its own right rather than a special case to
# paper over.  The z-scores below are therefore over the live coordinates.
live = se > 0
z_scores = (mc - stored)[live] / se[live]

print(f"z @ W   : max |MC - stored| = {np.abs(mc - stored).max():.2e}")
print(f"          live coordinates ({live.sum()}/{live.size}): "
      f"max |z| = {np.abs(z_scores).max():.1f} standard errors, "
      f"mean {z_scores.mean():+.2f}, std {z_scores.std():.2f}  (expect ~N(0,1))")
print(f"          dead coordinates ({(~live).sum()}): recomputed and stored both "
      f"exactly zero — {np.all(mc[~live] == 0) and np.all(stored[~live] == 0)}")
print(f"z @ W.T : max |MC - stored| = {np.abs(mc_T - stored).max():.2e}"
      f"   ->  wrong by O(1), not by noise")

z @ W   : max |MC - stored| = 1.35e-02
          live coordinates (155/192): max |z| = 2.7 standard errors, mean -0.13, std 0.96  (expect ~N(0,1))
          dead coordinates (37): recomputed and stored both exactly zero — True
z @ W.T : max |MC - stored| = 5.78e+00   ->  wrong by O(1), not by noise


In [9]:
hi = float(max(stored.max(), mc.max(), mc_T.max())) * 1.05
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=[0, hi], y=[0, hi], mode="lines", name="exact agreement",
    line=dict(color=INK_SOFT, width=1, dash="dot"), hoverinfo="skip", showlegend=False))
for vals, colour, label in [(mc, C_BLUE, "z @ W  (correct)"),
                            (mc_T, C_ORANGE, "z @ W.T  (wrong)")]:
    fig.add_trace(go.Scatter(
        x=stored.ravel(), y=vals.ravel(), mode="markers", name=label,
        marker=dict(color=colour, size=6, opacity=0.7,
                    line=dict(width=1, color="#fcfcfb")),   # 1px surface ring
        hovertemplate=f"{label}<br>stored %{{x:.4f}}<br>recomputed %{{y:.4f}}<extra></extra>"))
fig.add_annotation(x=hi * 0.72, y=hi * 0.72, text="identity", showarrow=False,
                   font=dict(color=INK_SOFT, size=11), textangle=-45, yshift=12)
style(fig, f"Recomputing F(W) from the stored weights — {SMALL}", height=460,
      showlegend=True)
fig.update_xaxes(title="stored label F(W)", range=[0, hi])
fig.update_yaxes(title="recomputed by Monte Carlo", range=[0, hi],
                 scaleanchor="x", scaleratio=1)
fig.show()

The blue points sit on the identity line to within Monte-Carlo noise — the z-scores
above are standard normal, which is the quantitative version of that statement. The
orange points do not sit on it at all.

There is also an exact test that needs no sampling. At **depth 1** the answer is
analytic: the pre-activation of coordinate $j$ is $x \cdot W_{:,j} \sim N(0,
\|W_{:,j}\|^2)$, and $\mathbb{E}[\mathrm{relu}(N(0,\sigma^2))] = \sigma/\sqrt{2\pi}$, so

$$F(W)_j = \frac{\|W_{:,j}\|}{\sqrt{2\pi}} \qquad \text{(the column norm).}$$

The **column** norm is the fingerprint of the `z @ W` convention — a `z @ W.T`
forward would give the row norms instead. This is the cheapest possible check when
writing a new estimator, and it needs no ground-truth data at all.

<a id="4"></a>
## 4. The heavy tail of $F$

$F$ is a product of per-layer gains, so it is heavy-tailed across random $W$. This
is **not noise**: it is the large-deviation statistics of the finite-time Lyapunov
exponent, some products expanding and some contracting.

The distribution is shown on a log axis because a linear one is unreadable. Both
panels are the same numbers, aggregated differently, and the pair of ratios printed
below them is the thing to read: the tail is far more extreme **per coordinate** than
**per network**, because averaging over a network's coordinates washes out exactly
the concentration that §5 shows building up with depth. The gap between the two
ratios widens with width — at width 256 the per-network spread is only about 2×
while individual coordinates reach 26× their median.

Error metrics inherit this: a per-coordinate mean squared error is dominated by a
small fraction of coordinates, which is why §7's residual analysis is done per
coordinate rather than per network.

In [10]:
Fb = np.asarray(F, dtype=np.float64)
per_net = Fb.mean(axis=1)
positive = Fb[Fb > 0]

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
                    subplot_titles=("per coordinate", "per network (mean over coordinates)"))
fig.add_trace(go.Histogram(x=np.log10(positive), nbinsx=60, marker_color=C_BLUE,
                           marker_line_width=0, name="coordinates", showlegend=False,
                           hovertemplate="log10 F = %{x:.2f}<br>%{y} coords<extra></extra>"),
              row=1, col=1)
fig.add_trace(go.Histogram(x=np.log10(per_net), nbinsx=30, marker_color=C_ORANGE,
                           marker_line_width=0, name="networks", showlegend=False,
                           hovertemplate="log10 mean F = %{x:.2f}<br>%{y} nets<extra></extra>"),
              row=1, col=2)
style(fig, f"Distribution of F — {BIG}", height=380)
fig.update_xaxes(title="log₁₀ F  (nonzero coordinates)", row=1, col=1)
fig.update_xaxes(title="log₁₀ mean F", row=1, col=2)
fig.update_yaxes(title="count", row=1, col=1)
fig.update_annotations(font=dict(size=13, color=INK_SOFT))
fig.show()

zero_frac = float((Fb == 0).mean())
print(f"coordinates exactly zero : {100 * zero_frac:.2f}%")
print(f"nonzero coordinates      : median {np.median(positive):.4f}   max {positive.max():.3f}"
      f"   -> {positive.max() / np.median(positive):.0f}x the median")
print(f"per-network means        : median {np.median(per_net):.4f}   max {per_net.max():.3f}"
      f"   -> {per_net.max() / np.median(per_net):.1f}x the median")

coordinates exactly zero : 17.72%
nonzero coordinates      : median 0.2004   max 5.254   -> 26x the median
per-network means        : median 0.3831   max 0.740   -> 1.9x the median


<a id="5"></a>
## 5. The ReLU cone collapses with depth

A layer keeps a coordinate alive only if some input direction leaves it positive.
Because `relu` is **positively homogeneous** ($\mathrm{relu}(cz) = c\,\mathrm{relu}(z)$
for $c>0$), rescaling the weights never changes *which* coordinates those are — so
this is a property of the geometry, not of the initialisation scale. No choice of
variance fixes it.

`part='all_layers'` returns the per-layer mean stack, which lets us watch the
collapse happen. Here magnitude is the job, so the colour is a **single hue,
light → dark**; near-zero recedes toward the surface, which is exactly what we want
to see.

In [11]:
_, all_layers = load_data_as_sequence(BIG, local=True, part="all_layers")
AL = np.asarray(all_layers, dtype=np.float64)
assert np.array_equal(AL[:, -1], Fb), "the last layer of the stack must be F itself"

fig = go.Figure(go.Heatmap(
    z=AL[0].T, colorscale=SEQ_BLUE, zmin=0,
    colorbar=dict(title="E[z]", thickness=12, outlinewidth=0),
    hovertemplate="layer %{x}<br>coordinate %{y}<br>mean %{z:.4f}<extra></extra>"))
style(fig, f"Mean activation by layer, one network — {BIG}, network 0", height=400)
fig.update_xaxes(title="layer")
fig.update_yaxes(title="coordinate")
fig.show()

Coordinates go dark and stay dark: once a coordinate is dead it never revives,
because the reachable set of the chain lies entirely in the halfspace where its
pre-activation is non-positive.

Averaged over networks, two curves tell the story. The mean over **all**
coordinates drifts down as coordinates die — but the mean over the **surviving**
coordinates goes *up*. The dynamics conserve mass while shrinking the support: this
is the concentration that produces the heavy tail in §4.

In [12]:
dead_by_layer = 100 * (AL == 0).mean(axis=(0, 2))
mean_all = AL.mean(axis=(0, 2))
mean_live = np.array([AL[:, l][AL[:, l] > 0].mean() if (AL[:, l] > 0).any() else 0.0
                      for l in range(depth)])
layers = np.arange(1, depth + 1)

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.13,
                    subplot_titles=("coordinates that are exactly zero",
                                    "mean activation: all vs surviving"))
fig.add_trace(go.Scatter(x=layers, y=dead_by_layer, mode="lines+markers",
                         line=dict(color=C_BLUE, width=2), marker=dict(size=6),
                         showlegend=False,
                         hovertemplate="layer %{x}<br>%{y:.2f}% dead<extra></extra>"),
              row=1, col=1)
for vals, colour, label in [(mean_all, C_BLUE, "all coordinates"),
                            (mean_live, C_ORANGE, "surviving only")]:
    fig.add_trace(go.Scatter(x=layers, y=vals, mode="lines", name=label,
                             line=dict(color=colour, width=2),
                             hovertemplate=f"{label}<br>layer %{{x}}<br>%{{y:.4f}}<extra></extra>"),
                  row=1, col=2)
    # Direct labels, so identity is never carried by colour alone.
    fig.add_annotation(x=depth, y=vals[-1], text=label, xref="x2", yref="y2",
                       showarrow=False, xanchor="right", yshift=14,
                       font=dict(color=colour, size=11))
style(fig, f"The cone collapsing with depth — {BIG}, {info[BIG]['num_points']:,} networks",
      height=390, showlegend=False)
fig.update_xaxes(title="layer", row=1, col=1)
fig.update_xaxes(title="layer", row=1, col=2)
fig.update_yaxes(title="% dead", row=1, col=1)
fig.update_yaxes(title="mean activation", row=1, col=2)
fig.update_annotations(font=dict(size=13, color=INK_SOFT))
fig.show()

print(f"final layer: {dead_by_layer[-1]:.2f}% of coordinates dead, "
      f"mean over all {mean_all[-1]:.4f} vs mean over surviving {mean_live[-1]:.4f}")

final layer: 17.72% of coordinates dead, mean over all 0.4145 vs mean over surviving 0.5038


<a id="6"></a>
## 6. Width scaling at fixed depth

Deadness is governed by **depth**, and is nearly independent of width once the
width is not small. The per-layer survival probability of the cone self-averages as
$d$ grows; only narrow networks die faster, from finite-size fluctuations.

That is why the dataset ladder holds depth at the competition's 32 and sweeps
width: it varies the *dimension* while holding the *regime* fixed. (With only the
core dataset generated there is one point per depth and this reads as a single
marker — generate the ladder with `--whest` to fill it in.)

In [13]:
rows = []
for n, i in info.items():
    rows.append({"dataset": n, "width": i["width"], "depth": i["depth"],
                 "dead_coords": 100 * i["dead_coordinate_fraction"]})
scaling = pd.DataFrame(rows).sort_values(["depth", "width"])

# Categorical hues are assigned in fixed order and never cycled: three slots are what
# validate as colourblind-safe for a scatter/line form with all pairs on screen.  The
# shipped ladder has two depths, so this is not a constraint in practice -- but if a
# fourth appeared it would need faceting rather than a fourth hue.
SLOTS = [C_BLUE, C_ORANGE, C_AQUA]
depth_groups = list(scaling.groupby("depth"))
if len(depth_groups) > len(SLOTS):
    print(f"note: {len(depth_groups)} depths present; showing the first {len(SLOTS)}. "
          "Facet the rest rather than adding hues.")
    depth_groups = depth_groups[: len(SLOTS)]

fig = go.Figure()
for k, (d_val, grp) in enumerate(depth_groups):
    colour = SLOTS[k]
    fig.add_trace(go.Scatter(
        x=grp["width"], y=grp["dead_coords"], mode="lines+markers",
        name=f"depth {d_val}", line=dict(color=colour, width=2),
        marker=dict(size=9, line=dict(width=1, color="#fcfcfb")),
        hovertemplate=f"depth {d_val}<br>width %{{x}}<br>%{{y:.1f}}% dead<extra></extra>"))
    fig.add_annotation(x=math.log10(grp["width"].iloc[-1]), y=grp["dead_coords"].iloc[-1],
                       text=f"depth {d_val}", showarrow=False, xanchor="left", xshift=10,
                       font=dict(color=colour, size=11))
style(fig, "Dead coordinates vs width, by depth", height=400, showlegend=False)
fig.update_xaxes(title="width", type="log",
                 tickvals=sorted(scaling["width"].unique()),
                 ticktext=[str(w) for w in sorted(scaling["width"].unique())])
fig.update_yaxes(title="% of coordinates exactly zero", rangemode="tozero")
fig.show()

scaling.pivot_table(index="width", columns="depth", values="dead_coords").round(2)

depth,8,32
width,,
8,19.85,NaN
16,6.51,38.41
32,NaN,30.16
64,NaN,25.48
128,NaN,22.09
256,0.24,17.72


<a id="7"></a>
## 7. The cheap baseline and its residual

Every dataset ships a cheap deterministic estimate alongside the truth, reachable
as `part='start'`. It is a fixed-quadrature **unscented transform**: represent
$N(0, I_d)$ by the $2d$ sigma points $\pm r\,e_i$ — the rows of $[rI; -rI]$ —
push them through the network, and average.

The radius is not $\sqrt{d}$. Because relu chains are positively homogeneous of
degree one, the per-coordinate mean is set by the **first** radial moment,

$$r = \mathbb{E}\|x\| = \sqrt{2}\,\frac{\Gamma((d+1)/2)}{\Gamma(d/2)},$$

whereas $\sqrt{d}$ matches only $\mathbb{E}\|x\|^2$ and so overestimates by
$1 + 1/(4d)$ — a bias that compounds with depth.

No random rotations are used, so `UT_fixed` is a **deterministic function of $W$**,
and therefore so is the residual

$$R(W) = F(W) - \mathrm{UT_{fixed}}(W).$$

That matters: correcting this baseline is a well-posed regression problem, not a
variance-reduction problem. A correcting function provably exists.

In [14]:
_, ut = load_data_as_sequence(BIG, local=True, part="start")
UT = np.asarray(ut, dtype=np.float64)
R = Fb - UT
dead = Fb == 0

radius = math.sqrt(2.0) * math.exp(math.lgamma((width + 1) / 2) - math.lgamma(width / 2))
print(f"sigma-point radius E||x|| = {radius:.4f}   (sqrt(width) = {math.sqrt(width):.4f}, "
      f"ratio {math.sqrt(width) / radius:.6f})")
print(f"UT final_layer_mse        = {((UT - Fb) ** 2).mean(axis=1).mean():.3e}"
      f"   (metadata: {info[BIG]['ut_final_layer_mse']:.3e})")

sigma-point radius E||x|| = 15.9844   (sqrt(width) = 16.0000, ratio 1.000977)
UT final_layer_mse        = 2.837e-05   (metadata: 2.837e-05)


In [15]:
hi = float(max(Fb.max(), UT.max())) * 1.05
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.13,
                    subplot_titles=("estimate vs truth", "signed residual R = F − UT"))
fig.add_trace(go.Scatter(x=[0, hi], y=[0, hi], mode="lines",
                         line=dict(color=INK_SOFT, width=1, dash="dot"),
                         showlegend=False, hoverinfo="skip"), row=1, col=1)
fig.add_trace(go.Scatter(
    x=Fb.ravel(), y=UT.ravel(), mode="markers", showlegend=False,
    marker=dict(color=C_BLUE, size=4, opacity=0.35),
    hovertemplate="F %{x:.4f}<br>UT %{y:.4f}<extra></extra>"), row=1, col=1)
fig.add_trace(go.Histogram(
    x=R.ravel(), nbinsx=80, marker_color=C_BLUE, marker_line_width=0,
    showlegend=False, hovertemplate="R = %{x:.4f}<br>%{y}<extra></extra>"), row=1, col=2)
style(fig, f"The cheap baseline and what it misses — {BIG}", height=400)
fig.update_xaxes(title="truth F(W)", row=1, col=1, range=[0, hi])
fig.update_yaxes(title="UT_fixed(W)", row=1, col=1, range=[0, hi],
                 scaleanchor="x", scaleratio=1)
fig.update_xaxes(title="residual", row=1, col=2)
fig.update_yaxes(title="count", type="log", row=1, col=2)
fig.update_annotations(font=dict(size=13, color=INK_SOFT))
fig.show()

print(f"residual: mean {R.mean():+.2e}  std {R.std():.2e}  "
      f"max |R| {np.abs(R).max():.4f}")

residual: mean -7.02e-04  std 5.28e-03  max |R| 0.0285


### Where the error actually lives

The residual is centred near zero and heavy-tailed — locating a large error is easy,
but its **sign** is what a corrector needs, and the mean being ~0 says the sign is
not implied by the magnitude.

There is one piece of structure that is free, though. Dead coordinates are a *cone*
property: if the reachable set lies in the halfspace where coordinate $j$ is
non-positive, then the sigma points' images lie in that same set, so any
point-propagation estimator returns **exactly** zero there. The baseline gets every
dead coordinate exactly right, at no cost — so all of its error lives on the
surviving coordinates.

In [16]:
mse_dead = float((R[dead] ** 2).mean()) if dead.any() else 0.0
mse_live = float((R[~dead] ** 2).mean())
share = 100 * (R[dead] ** 2).sum() / (R ** 2).sum() if dead.any() else 0.0

fig = go.Figure(go.Bar(
    x=[f"dead ({100 * dead.mean():.1f}% of coords)", f"live ({100 * (~dead).mean():.1f}%)"],
    y=[mse_dead, mse_live], marker_color=[C_AQUA, C_BLUE], marker_line_width=0,
    width=0.5, text=[f"{mse_dead:.2e}", f"{mse_live:.2e}"], textposition="outside",
    textfont=dict(color=INK_SOFT, size=12),
    hovertemplate="%{x}<br>MSE %{y:.3e}<extra></extra>"))
style(fig, f"UT error by coordinate type — {BIG}", height=360, showlegend=False)
fig.update_yaxes(title="mean squared error", rangemode="tozero")
fig.show()

print(f"MSE on dead coordinates : {mse_dead:.3e}   ({share:.1f}% of total squared error)")
print(f"MSE on live coordinates : {mse_live:.3e}")
print("\nSo the entire estimation problem is on the surviving coordinates; a learned")
print("model that regresses F directly must *learn* the zeros that UT gets for free.")

MSE on dead coordinates : 0.000e+00   (0.0% of total squared error)
MSE on live coordinates : 3.448e-05

So the entire estimation problem is on the surviving coordinates; a learned
model that regresses F directly must *learn* the zeros that UT gets for free.


<a id="8"></a>
## 8. What a learned model has to beat

Two numbers bracket the useful range for every dataset:

- the **label noise floor** `label_mc_se2` — below this, differences are not
  measurable from this dataset;
- the **baseline error** `ut_final_layer_mse` — above this, a model is not yet
  worth preferring to sixteen matrix multiplications.

Both are plotted on one logarithmic axis so the gap is legible; the shaded band
between them is the room a learned model has to work in. The official datasets have
ground truth at $10^9$ samples per network, which is why their floors sit orders of
magnitude lower.

In [17]:
order = inventory.sort_values("ut_mse")
fig = go.Figure()
for _, row in order.iterrows():
    fig.add_trace(go.Scatter(
        x=[row["label_mc_se2"], row["ut_mse"]], y=[row["dataset"]] * 2,
        mode="lines", line=dict(color=GRID, width=8), showlegend=False,
        hoverinfo="skip"))
for col, colour, label in [("label_mc_se2", C_AQUA, "label noise floor"),
                           ("ut_mse", C_BLUE, "UT baseline error")]:
    fig.add_trace(go.Scatter(
        x=order[col], y=order["dataset"], mode="markers", name=label,
        marker=dict(color=colour, size=11, line=dict(width=1.5, color="#fcfcfb")),
        hovertemplate=f"{label}<br>%{{y}}<br>%{{x:.3e}}<extra></extra>"))
style(fig, "The room a learned model has, per dataset", height=120 + 42 * len(order),
      showlegend=True, legend=dict(orientation="h", y=1.06, x=0, font_color=INK_SOFT))
fig.update_xaxes(title="mean squared error (log scale)", type="log")
fig.update_yaxes(title=None, autorange="reversed")
fig.show()

order[["dataset", "networks", "label_mc_se2", "ut_mse", "headroom"]].style.format(
    {"label_mc_se2": "{:.2e}", "ut_mse": "{:.3e}", "headroom": "{:,.0f}x",
     "networks": "{:,d}"}).hide(axis="index")

dataset,networks,label_mc_se2,ut_mse,headroom
whest_w256_d32,11,3.66e-11,2.837e-05,"775,091x"
whest_w128_d32,44,4.04e-09,9.954e-05,"24,628x"
whest_w256_d8,45,1.74e-10,1.310e-04,"751,034x"
whest_w64_d32,178,6.60e-09,2.919e-04,"44,232x"
whest_w32_d32,701,1.14e-08,5.135e-04,"45,131x"
whest_w16_d32,"2,700",8.65e-09,5.614e-04,"64,907x"
whest_w16_d8,"10,000",1.89e-08,1.387e-03,"73,436x"
whest_w8_d8,"10,000",1.50e-06,2.167e-03,"1,440x"


### Reading these numbers honestly

- **Report raw MSE.** $F$ is heavy-tailed, so training in $\log(1+F)$ space is
  sensible — but $R^2$ on log targets flatters by roughly three orders of
  magnitude. The contest scores raw MSE; so should you.
- **Small $N$ wobbles.** The same fixed-UT baseline scores 2.837e-5 on the
  11-network dataset and 4.340e-5 on the 56-network one. Eleven networks give ~30%
  standard error on a measured MSE; 224 give ~7%. Each dataset carries a
  `statistical_note` saying so, and for a properly powered evaluation at the
  competition geometry you want the full 1,000-network official split from the
  HuggingFace Hub rather than these subsets.
- **Per-layer means are labels, never inputs.** The estimator contract passes only
  the weights, the geometry, a seed and a name — no means of any kind. Being handed
  the per-layer means would collapse the depth-32 error compounding into a
  single-layer problem and gut the task, which is why
  `load_data_as_sequence` refuses `label_every_step=True` for this family.

---

**Further reading:** `README_whest.md` in the repository root — the full reference
for these datasets, including the storage layout, the official-data provenance and
attribution, and the metadata schema.